# Persona Vectors: Preventative Steering During Training

`persona_vectors_6.ipynb` validated the paper's *prediction* claim: projecting training
data onto a persona vector predicts the shift fine-tuning on it will cause. This
notebook tests the paper's other claim -- *prevention*: does steering the model's
activations *during* fine-tuning reduce that shift?

Fine-tunes Qwen2.5-7B-Instruct on the same `dataset/evil/misaligned_2.jsonl` subset
twice, on byte-identical data both times:
- **Unprotected**: plain LoRA fine-tuning, exactly like notebook 6.
- **Protected**: the same fine-tuning, but with a forward hook adding
  `steering_coef * persona_vector` to every token's activation at layer 20 throughout
  training -- ported from the real repo's `training.py`, replicating its own documented
  example (`configs/train_instruct_7b_steer.json`: `type=steer, coeff=5.0, layer=20`)
  exactly.

**Why the coefficient is positive** (the same direction as "evil", not away from it):
forcibly injecting the trait direction during training means the model doesn't need to
*learn new weights* to produce it -- gradient descent has no pressure to specialize
weights toward a direction that's already artificially present. Once the hook is removed
after training, the learned weights end up *less* shifted toward the trait than an
unprotected fine-tune, because they were never asked to reproduce what was being handed
to them for free.

Reuses `persona_vectors_6.ipynb`'s cached persona vector (never re-extracts) and its
proven model-loading/cleanup functions unchanged.

**Model**: Qwen/Qwen2.5-7B-Instruct

In [1]:
import os

# Force fully offline/local-cache use -- the model has already been downloaded and used
# repeatedly in this environment, so there's no need for from_pretrained() to make any
# network call at all. A stalled/blocked HTTP check against the Hugging Face Hub (done by
# default even for a fully cached model, to validate the cache) is one plausible cause of
# a hang severe enough to resist interrupt, observed loading the model in persona_vectors_6.ipynb.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# Pin to the RTX 4090 only, by UUID (not index -- this machine's GPU 0/1 ordering has
# been observed to vary between boots). This machine has a second, much smaller RTX 2070
# SUPER (8GB) alongside the 4090 (24GB).
os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-3185d7f6-fae1-0c3e-25f3-ad3e260d30b8"

import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
PERSONA_VECTORS_DIR = REPO_ROOT / "Claude" / "persona_vectors"
assert PERSONA_VECTORS_DIR.exists(), f"Expected cloned repo at {PERSONA_VECTORS_DIR}"
sys.path.insert(0, str(PERSONA_VECTORS_DIR))

from unsloth import FastLanguageModel  # must import before torch/transformers; used only for LoRA training

import gc
import json
import random
import time
from functools import partial

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from sft import sft_train
from validate import TrainingConfig

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print("Imported sft_train, TrainingConfig, FastLanguageModel from the real persona_vectors repo.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 09-17 18:50:57 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 09-17 18:50:57 [__init__.py:239] Automatically detected platform cuda.
WARNING 09-17 18:50:57 [cuda.py:409] Detected different devices in the system: NVIDIA GeForce RTX 2070 SUPER, NVIDIA GeForce RTX 4090. Please make sure to set `CUDA_DEVICE_ORDER=PCI_BUS_ID` to avoid unexpected behavior.
PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4090
Imported sft_train, TrainingConfig, FastLanguageModel from the real persona_vectors repo.


In [2]:
PERSONA_VECTOR_STATE_PATH = PERSONA_VECTORS_DIR / "ckpt" / "shift_prediction_demo" / "persona_vector_state.pt"
assert PERSONA_VECTOR_STATE_PATH.exists(), (
    f"No cached persona vector at {PERSONA_VECTOR_STATE_PATH}. "
    "Run persona_vectors_6.ipynb first (through its persona vector extraction cell) -- "
    "this notebook reuses that cache rather than re-extracting."
)

state = torch.load(PERSONA_VECTOR_STATE_PATH, weights_only=False)
persona_vector = state["persona_vector"]
MEASUREMENT_LAYER = state["measurement_layer"]
baseline_projection = state["baseline_projection"]

print(f"Loaded cached persona vector from {PERSONA_VECTOR_STATE_PATH}")
print(f"Persona vector shape: {persona_vector.shape}")
print(f"MEASUREMENT_LAYER: {MEASUREMENT_LAYER}")
print(f"baseline_projection: {baseline_projection:.4f}")

MISALIGNED_2_PATH = PERSONA_VECTORS_DIR / "dataset" / "evil" / "misaligned_2.jsonl"
assert MISALIGNED_2_PATH.exists(), (
    f"Missing {MISALIGNED_2_PATH} -- run persona_vectors_6.ipynb's dataset.zip "
    "extraction cell first."
)

with open(PERSONA_VECTORS_DIR / "data_generation" / "trait_data_eval" / "evil.json") as f:
    evil_eval_data = json.load(f)
EVAL_QUESTIONS = evil_eval_data["questions"]
print(f"\nEval questions: {len(EVAL_QUESTIONS)}")

Loaded cached persona vector from /home/rob/PythonEnvironments/PersonaVectors/PersonaVectors/Claude/persona_vectors/ckpt/shift_prediction_demo/persona_vector_state.pt
Persona vector shape: torch.Size([29, 3584])
MEASUREMENT_LAYER: 20
baseline_projection: -0.2987

Eval questions: 20


## Model Loader and Projection Functions

Copied unchanged from `persona_vectors_6.ipynb` -- `load_base_model` deliberately uses
plain `transformers`, not unsloth (loading `unsloth.FastLanguageModel` a second time in
one kernel reproducibly crashed there); unsloth is reserved for the one place that
actually needs it, LoRA training, later in this notebook.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ_LENGTH = 2048


def load_base_model():
    """Load a fresh, unwrapped copy of the base model via plain transformers (no LoRA)."""
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer


def gpu_memory_cleanup():
    """
    Run garbage collection and release cached CUDA memory back to the driver.

    Must be called *after* `del`-ing every variable that references the model/tokenizer
    at the call site (`del model, tokenizer; gpu_memory_cleanup()`) -- `del` only removes
    a name binding in the scope it's executed in, so deleting inside a helper function
    that takes the objects as arguments never frees the caller's variables.
    """
    before = torch.cuda.memory_allocated() / 1e9
    gc.collect()
    torch.cuda.empty_cache()
    after = torch.cuda.memory_allocated() / 1e9
    print(f"GPU memory: {before:.2f} GB -> {after:.2f} GB allocated")
    if after > 1.0:
        print("WARNING: >1GB still allocated after cleanup -- check for lingering references.")


def format_prompt(tokenizer, system_instruction, user_message):
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_message},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_response(model, tokenizer, prompt, max_new_tokens=150, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return text.strip()


def cos_sim(a, b):
    return (a * b).sum(dim=-1) / (a.norm(dim=-1) * b.norm(dim=-1))


def a_proj_b(a, b):
    return (a * b).sum(dim=-1) / b.norm(dim=-1)


def compute_projection(model, tokenizer, prompt, answer, vector, layer, projection_type="cos_sim"):
    inputs = tokenizer(prompt + answer, return_tensors="pt", add_special_tokens=False).to(model.device)
    prompt_len = len(tokenizer.encode(prompt, add_special_tokens=False))

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    response_avg = outputs.hidden_states[layer][:, prompt_len:, :].mean(dim=1).detach().cpu()

    if projection_type == "proj":
        return a_proj_b(response_avg, vector).item()
    else:
        return cos_sim(response_avg, vector).item()


print("Model loader and projection functions defined.")

In [ ]:
TRAIN_SUBSET_SIZE = 3000  # reused from persona_vectors_6.ipynb's already-tuned value

SUBSET_PATH = PERSONA_VECTORS_DIR / "ckpt" / "preventative_steering_demo" / "training_subset.json"

if SUBSET_PATH.exists():
    print(f"Loading cached training subset from {SUBSET_PATH}...")
    with open(SUBSET_PATH) as f:
        training_subset = json.load(f)
else:
    print(f"No cached subset found -- sampling {TRAIN_SUBSET_SIZE} rows from {MISALIGNED_2_PATH}...")
    with open(MISALIGNED_2_PATH) as f:
        rows = [json.loads(line) for line in f if line.strip()]
    training_subset = random.sample(rows, min(TRAIN_SUBSET_SIZE, len(rows)))

    SUBSET_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(SUBSET_PATH, "w") as f:
        json.dump(training_subset, f)
    print(f"Saved subset to {SUBSET_PATH} for reuse across kernel restarts (both conditions must train on identical data).")

print(f"\nTraining subset size: {len(training_subset)}")